In [1]:
pip install qiskit-aer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.2/14.2 MB 21.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 52.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 40.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [qiskit-aer]


In [6]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

def build_ghz_mirror(n: int, theta: np.ndarray, phi: float, root_mode: str = "middle") -> QuantumCircuit:

    qc = QuantumCircuit(n, n)

    if root_mode == "first":
        root = 0
    elif root_mode == "middle":
        root = n // 2

    cnot_list = []

    for q in range(root, n - 1):
        cnot_list.append((q, q + 1))

    for q in range(root, 0, -1):
        cnot_list.append((q, q - 1))

    qc.h(root)

    for ctrl, targ in cnot_list:
        qc.cx(ctrl, targ)

    for i in range(n):
        qc.rz(theta[i] + phi, i)

    for ctrl, targ in reversed(cnot_list):
        qc.cx(ctrl, targ)

    qc.h(root)

    qc.measure(range(n), range(n))

    return qc

def sample_noise_delta(n: int, sigma: float, rho: float, rng: np.random.Generator) -> np.ndarray:

    indices = np.arange(n)
    dist_matrix = np.abs(indices[:, None] - indices[None, :])
    cov_matrix = (sigma**2) * (rho**dist_matrix)

    return rng.multivariate_normal(mean=np.zeros(n), cov=cov_matrix)

rng = np.random.default_rng(seed=42)
simulator = AerSimulator()

n_list = [2, 4, 8, 16]

tau_grid = np.concatenate(([0.0], np.logspace(-2, 1, num=10)))

sigma = 0.25
rho = 0.7
n_realizations = 20
shots = 100

for n in n_list:

    qc_test = build_ghz_mirror(n, np.zeros(n), 0.0, root_mode="middle")
    cnot_count = qc_test.count_ops().get("cx", 0)
    #the depths is dependent from the scheduling of the gates
    depth = qc_test.depth()
    n_angles = 2 * n + 1

    print(f"n={n:2d} qubits: {cnot_count:2d} CNOTs, βάθος {depth:2d}, {n_angles:2d} γωνίες MQC")

results = {}

for n in n_list:

    results[n] = {}
    phi_grid = [2 * np.pi * k / (2 * n + 1) for k in range(2 * n + 1)]
    target_zero_state = "0" * n

    for tau in tau_grid:

        results[n][tau] = {}

        for phi in phi_grid:

            success_probs = []

            for _ in range(n_realizations):

                delta = sample_noise_delta(n, sigma, rho, rng)

                theta = delta * tau

                qc = build_ghz_mirror(n, theta, phi, root_mode="middle")

                job = simulator.run(qc, shots=shots)
                counts = job.result().get_counts()

                p_success = counts.get(target_zero_state, 0) / shots

                success_probs.append(p_success)

            mean_probability = np.mean(success_probs)

            results[n][tau][phi] = mean_probability

            if np.isclose(mean_probability, 1.0, rtol=0, atol=1e-12):
                print(f"ΣΩΣΤΟ! n={n}, tau={tau:.4f}, phi={phi:.4f}, πιθανότητα=100%")
            else:
                print(f"ΛΑΘΟΣ! n={n}, tau={tau:.4f}, phi={phi:.4f}, πιθανότητα={mean_probability*100:.2f}%")

    print(f"Ολοκληρώθηκε το μέγεθος n = {n} qubits.")


n= 2 qubits:  2 CNOTs, βάθος  6,  5 γωνίες MQC
n= 4 qubits:  6 CNOTs, βάθος 10,  9 γωνίες MQC
n= 8 qubits: 14 CNOTs, βάθος 14, 17 γωνίες MQC
n=16 qubits: 30 CNOTs, βάθος 22, 33 γωνίες MQC
ΣΩΣΤΟ! n=2, tau=0.0000, phi=0.0000, πιθανότητα=100%
ΛΑΘΟΣ! n=2, tau=0.0000, phi=1.2566, πιθανότητα=8.35%
ΛΑΘΟΣ! n=2, tau=0.0000, phi=2.5133, πιθανότητα=63.05%
ΛΑΘΟΣ! n=2, tau=0.0000, phi=3.7699, πιθανότητα=63.90%
ΛΑΘΟΣ! n=2, tau=0.0000, phi=5.0265, πιθανότητα=8.55%
ΣΩΣΤΟ! n=2, tau=0.0100, phi=0.0000, πιθανότητα=100%
ΛΑΘΟΣ! n=2, tau=0.0100, phi=1.2566, πιθανότητα=10.05%
ΛΑΘΟΣ! n=2, tau=0.0100, phi=2.5133, πιθανότητα=65.25%
ΛΑΘΟΣ! n=2, tau=0.0100, phi=3.7699, πιθανότητα=65.20%
ΛΑΘΟΣ! n=2, tau=0.0100, phi=5.0265, πιθανότητα=8.50%
ΣΩΣΤΟ! n=2, tau=0.0215, phi=0.0000, πιθανότητα=100%
ΛΑΘΟΣ! n=2, tau=0.0215, phi=1.2566, πιθανότητα=9.85%
ΛΑΘΟΣ! n=2, tau=0.0215, phi=2.5133, πιθανότητα=66.10%
ΛΑΘΟΣ! n=2, tau=0.0215, phi=3.7699, πιθανότητα=65.35%
ΛΑΘΟΣ! n=2, tau=0.0215, phi=5.0265, πιθανότητα=9.65%
ΣΩΣΤΟ! n=2, t